# Amazon ML Challenge 2026 — FINAL SAFE INFERENCE

This is a clean final notebook for the Business Entity Resolution challenge.

**Design goals**

- Use the supplied `amazon_ml_solution` modules.
- Do not depend on notebook execution order from the old 179-cell notebook.
- Do not load TEST S2/S3 completely into RAM.
- Keep working memory below the 20 GB constraint.
- Use the validated blocking/scoring configuration:
  - cheap threshold = 0.10
  - detailed threshold = 0.70
  - score = 0.50 name + 0.30 address + 0.15 address-number + 0.05 country
- Use checkpoints so a long inference cell can be safely resumed.
- Every executable cell has a Markdown explanation immediately before it.

**IMPORTANT:** Run cells in order. Do not use Run All.

In [ ]:
# ============================================================
# CELL 1 — ROBUST KAGGLE SETUP + MODULE LOADING
# ============================================================

import os
import sys
import gc
import csv
import time
import json
import heapq
import pickle
import importlib
from pathlib import Path
from collections import defaultdict

print("=" * 72)
print("CELL 1 — ROBUST SETUP")
print("=" * 72)

# ------------------------------------------------------------
# Locate the supplied student_resource directory.
# We do NOT assume the current working directory.
# ------------------------------------------------------------

ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource"),
    Path("/kaggle/input/amazon-ml-challenge-2026-student-resource/student_resource"),
]

ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "amazon_ml_solution" / "src" / "normalize.py").exists()),
    None
)

if ROOT is None:
    # Last-resort search only inside /kaggle/input.
    matches = list(
        Path("/kaggle/input").glob(
            "**/student_resource/amazon_ml_solution/src/normalize.py"
        )
    )
    if matches:
        ROOT = matches[0].parents[3]

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate student_resource/amazon_ml_solution/src/normalize.py. "
        "This notebook must be run in the supplied Kaggle competition environment."
    )

ROOT = Path(ROOT)
TRAIN_DIR = ROOT / "dataset" / "train"
TEST_DIR = ROOT / "dataset" / "test"
SOLUTION_DIR = ROOT / "amazon_ml_solution"
SRC_DIR = SOLUTION_DIR / "src"
UTILS_DIR = ROOT / "utils"

WORK_DIR = Path("/kaggle/working/amazon_ml_final")
OUTPUT_DIR = WORK_DIR / "output"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

# ------------------------------------------------------------
# THE IMPORTANT FIX:
# Put the exact source directory on sys.path BEFORE importing.
# Also invalidate the import cache.
# ------------------------------------------------------------

src_string = str(SRC_DIR.resolve())

if src_string not in sys.path:
    sys.path.insert(0, src_string)

importlib.invalidate_caches()

required_modules = [
    SRC_DIR / "normalize.py",
    SRC_DIR / "candidate_generator.py",
    SRC_DIR / "similarity.py",
]

missing = [str(p) for p in required_modules if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Required supplied solution files are missing:\n" + "\n".join(missing)
    )

# Import the supplied modules ONLY after sys.path is correct.
from normalize import normalize_business_name, normalize_address
from candidate_generator import make_name_keys, make_address_keys
from similarity import calculate_similarity

print("ROOT     :", ROOT)
print("SRC_DIR  :", SRC_DIR)
print("TEST_S1  :", TEST_S1.exists())
print("TEST_S2  :", TEST_S2.exists())
print("TEST_S3  :", TEST_S3.exists())
print()
print("normalize.py          : OK")
print("candidate_generator.py: OK")
print("similarity.py         : OK")
print()
print("✅ CELL 1 COMPLETE — IMPORTS ARE READY")

## Cell 2 — Verify the supplied files and imported functions

This is a lightweight safety check. It does not load the large test files.

In [ ]:
# ============================================================
# CELL 2 — LIGHTWEIGHT PREFLIGHT CHECK
# ============================================================

print("=" * 72)
print("CELL 2 — PREFLIGHT")
print("=" * 72)

for name, path in {
    "TEST S1": TEST_S1,
    "TEST S2": TEST_S2,
    "TEST S3": TEST_S3,
}.items():
    assert path.exists(), f"{name} missing: {path}"
    print(f"{name:<8}: {path.stat().st_size / 1024**2:,.1f} MB")

# Check function calls on tiny examples.
example_name = "Koch Premier Newbury Inc."
example_address = "123 Main Street, Mumbai"

print()
print("Name normalization :", normalize_business_name(example_name))
print("Address normalization:", normalize_address(example_address))
print("Name keys          :", make_name_keys(example_name))
print("Address keys       :", make_address_keys(example_address))

print()
print("✅ CELL 2 COMPLETE")

## Cell 3 — Final matching configuration

These are the settings validated during development. No experimental
threshold sweep is performed here, so this cell is fast.

In [ ]:
# ============================================================
# CELL 3 — FINAL CONFIGURATION
# ============================================================

CHEAP_THRESHOLD = 0.10
FINAL_THRESHOLD = 0.70

# To keep the final output precise, retain only the best few detailed
# candidates per S1 from each source. Multiple genuine matches are still
# allowed when they are above FINAL_THRESHOLD.
TOP_K_PER_S1 = 5

# A blocking posting longer than this is ignored during the fast final
# pass. Exact normalized name/address matching is always retained.
#
# This is a RESOURCE-SAFETY setting, not a claim that every frequent
# blocking key is useless.
MAX_POSTING = 100

# A source row is allowed to produce at most this many candidates before
# detailed scoring. This prevents pathological common-name cases from
# exploding runtime.
MAX_CANDIDATES_PER_SOURCE_ROW = 150

# Each inference cell voluntarily stops after this many minutes.
# Rerunning the SAME cell resumes from its checkpoint.
CELL_TIME_LIMIT_MIN = 45

SCORE_WEIGHTS = {
    "name_similarity": 0.50,
    "address_similarity": 0.30,
    "address_number_match": 0.15,
    "country_match": 0.05,
}

CONFIG_PATH = WORK_DIR / "pipeline_config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "cheap_threshold": CHEAP_THRESHOLD,
        "final_threshold": FINAL_THRESHOLD,
        "top_k_per_s1": TOP_K_PER_S1,
        "max_posting": MAX_POSTING,
        "max_candidates_per_source_row": MAX_CANDIDATES_PER_SOURCE_ROW,
        "cell_time_limit_min": CELL_TIME_LIMIT_MIN,
        "score_weights": SCORE_WEIGHTS,
    }, f, indent=2)

print("Cheap threshold :", CHEAP_THRESHOLD)
print("Final threshold :", FINAL_THRESHOLD)
print("Top K           :", TOP_K_PER_S1)
print("Max posting     :", MAX_POSTING)
print("Row candidate cap:", MAX_CANDIDATES_PER_SOURCE_ROW)
print("Cell time limit :", CELL_TIME_LIMIT_MIN, "min")
print("Config           :", CONFIG_PATH)
print()
print("✅ CELL 3 COMPLETE")

## Cell 4 — Build the TEST S1 blocking index

Only TEST S1 is held in memory. TEST S2 and TEST S3 remain on disk.

The index stores integer row positions rather than duplicating complete S1
dictionaries inside every posting list.

This is the key memory-control step.

In [ ]:
# ============================================================
# CELL 4 — BUILD COMPACT TEST-S1 INDEX
# ============================================================

import re

print("=" * 72)
print("CELL 4 — BUILDING TEST S1 INDEX")
print("=" * 72)

INDEX_PATH = CHECKPOINT_DIR / "s1_index.pkl"

# If a complete index already exists, reuse it.
if INDEX_PATH.exists():
    print("Existing S1 index found. Loading:", INDEX_PATH)

    with open(INDEX_PATH, "rb") as f:
        index_state = pickle.load(f)

    s1_ids = index_state["s1_ids"]
    s1_rows = index_state["s1_rows"]
    name_index = index_state["name_index"]
    address_index = index_state["address_index"]
    exact_name = index_state["exact_name"]
    exact_address = index_state["exact_address"]

    print("Loaded S1 rows:", f"{len(s1_ids):,}")

else:
    t0 = time.time()

    s1_ids = []
    s1_rows = []

    # key -> list of integer S1 positions
    name_index = defaultdict(list)
    address_index = defaultdict(list)

    # Exact normalized lookup
    exact_name = defaultdict(list)
    exact_address = defaultdict(list)

    with open(TEST_S1, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")

        for n, row in enumerate(reader, start=1):
            sid = str(row.get("entity_id", "")).strip()

            if not sid:
                continue

            country = str(row.get("country", "")).strip().upper()
            name = str(row.get("business_name", ""))
            address = str(row.get("business_address", ""))

            pos = len(s1_ids)

            s1_ids.append(sid)
            s1_rows.append({
                "entity_id": sid,
                "business_name": name,
                "business_address": address,
                "country": country,
            })

            # Exact normalized indexes
            nn = normalize_business_name(name)
            aa = normalize_address(address)

            if nn:
                exact_name[(country, nn)].append(pos)

            if aa:
                exact_address[(country, aa)].append(pos)

            # Validated multi-key blocker
            for key_type, key_value in make_name_keys(name):
                if key_value:
                    name_index[(country, key_type, key_value)].append(pos)

            for key_type, key_value in make_address_keys(address):
                if key_value:
                    address_index[(country, key_type, key_value)].append(pos)

            if n % 250_000 == 0:
                print(
                    f"S1 rows: {n:,} | "
                    f"time={(time.time()-t0):.1f}s"
                )

    index_state = {
        "s1_ids": s1_ids,
        "s1_rows": s1_rows,
        "name_index": dict(name_index),
        "address_index": dict(address_index),
        "exact_name": dict(exact_name),
        "exact_address": dict(exact_address),
    }

    with open(INDEX_PATH, "wb") as f:
        pickle.dump(index_state, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("Index checkpoint written:", INDEX_PATH)

print()
print("S1 rows:", f"{len(s1_ids):,}")
print("Name keys:", f"{len(name_index):,}")
print("Address keys:", f"{len(address_index):,}")
print("Exact name keys:", f"{len(exact_name):,}")
print("Exact address keys:", f"{len(exact_address):,}")

gc.collect()

print()
print("✅ CELL 4 COMPLETE")

## Cell 5 — Fast 10,000-row pilot

This cell is deliberately small. It does **not** score fuzzy similarity.

It checks how many S1 candidates the final blocker produces and gives an
early runtime signal before the expensive full pass.

In [ ]:
# ============================================================
# CELL 5 — 10K ROW PILOT
# ============================================================

def source_candidates(row):
    country = str(row.get("country", "")).strip().upper()
    name = str(row.get("business_name", ""))
    address = str(row.get("business_address", ""))

    candidate_counts = defaultdict(int)

    # Exact matches are always included.
    nn = normalize_business_name(name)
    aa = normalize_address(address)

    for pos in exact_name.get((country, nn), []) if nn else []:
        candidate_counts[pos] += 100

    for pos in exact_address.get((country, aa), []) if aa else []:
        candidate_counts[pos] += 100

    # Multi-key blocker. Ignore pathological postings.
    for kt, kv in make_name_keys(name):
        posting = name_index.get((country, kt, kv), [])
        if len(posting) <= MAX_POSTING:
            for pos in posting:
                candidate_counts[pos] += 1

    for kt, kv in make_address_keys(address):
        posting = address_index.get((country, kt, kv), [])
        if len(posting) <= MAX_POSTING:
            for pos in posting:
                candidate_counts[pos] += 1

    if not candidate_counts:
        return []

    # Strongest key support first, with a hard safety cap.
    return sorted(
        candidate_counts.items(),
        key=lambda x: (-x[1], x[0])
    )[:MAX_CANDIDATES_PER_SOURCE_ROW]


def pilot(path, rows=10_000):
    t0 = time.time()
    total = 0
    candidate_total = 0
    max_candidates = 0
    zero = 0

    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")

        for row in reader:
            total += 1
            cands = source_candidates(row)
            n = len(cands)
            candidate_total += n
            max_candidates = max(max_candidates, n)

            if n == 0:
                zero += 1

            if total >= rows:
                break

    elapsed = time.time() - t0

    return {
        "rows": total,
        "avg_candidates": candidate_total / max(total, 1),
        "max_candidates": max_candidates,
        "zero_candidates": zero,
        "seconds": elapsed,
        "rows_per_second": total / max(elapsed, 1e-9),
    }


for path, label in [(TEST_S2, "S2"), (TEST_S3, "S3")]:
    result = pilot(path)
    print()
    print(label)
    print("-" * 50)
    for k, v in result.items():
        print(f"{k:20s}: {v}")

print()
print("✅ CELL 5 COMPLETE")
print("If the pilot is fast and candidate counts are reasonable, continue.")

## Cell 6 — Define the two-stage scorer

The cheap stage uses only normalized strings and address numbers.
The expensive stage uses the supplied `calculate_similarity()` implementation.

The expensive scorer is therefore limited to candidates that first pass the
cheap structural filter.

In [ ]:
# ============================================================
# CELL 6 — TWO-STAGE SCORER
# ============================================================

def address_numbers_from_normalized(address):
    if not address:
        return set()
    return {
        x for x in address.split()
        if x.isdigit()
    }


def cheap_score_from_normalized(
    s1_name,
    s1_addr,
    s2_name,
    s2_addr,
    key_count
):
    exact_name = bool(s1_name) and s1_name == s2_name
    exact_address = bool(s1_addr) and s1_addr == s2_addr

    n1 = address_numbers_from_normalized(s1_addr)
    n2 = address_numbers_from_normalized(s2_addr)

    number_match = bool(n1 and n2 and (n1 & n2))

    c1 = s1_name.replace(" ", "")
    c2 = s2_name.replace(" ", "")

    prefix6 = (
        len(c1) >= 6 and len(c2) >= 6 and
        c1[:6] == c2[:6]
    )

    prefix8 = (
        len(c1) >= 8 and len(c2) >= 8 and
        c1[:8] == c2[:8]
    )

    return (
        1.00 * exact_name
        + 1.00 * exact_address
        + 0.70 * number_match
        + 0.50 * prefix8
        + 0.30 * prefix6
        + 0.05 * min(key_count, 10)
    )


def detailed_score(s1_row, s2_row):
    features = calculate_similarity(s1_row, s2_row)

    score = (
        SCORE_WEIGHTS["name_similarity"] * features["name_similarity"]
        + SCORE_WEIGHTS["address_similarity"] * features["address_similarity"]
        + SCORE_WEIGHTS["address_number_match"] * features["address_number_match"]
        + SCORE_WEIGHTS["country_match"] * features["country_match"]
    )

    return score


print("Cheap scorer: ready")
print("Detailed scorer: supplied similarity.py + validated weights")
print("Cheap threshold:", CHEAP_THRESHOLD)
print("Final threshold:", FINAL_THRESHOLD)
print("✅ CELL 6 COMPLETE")

## Cell 7 — Full TEST S2 inference with checkpoint/resume

This cell processes TEST S2 in a streaming fashion.

It does not load the full file into RAM.

**Resume behavior:** if the cell reaches the 45-minute guard, rerun the
same cell. It resumes from the saved checkpoint instead of starting S2 over.

Only the best `TOP_K_PER_S1` candidates are retained in memory for S2.

In [ ]:
# ============================================================
# CELL 7 — STREAM TEST S2
# ============================================================

def run_source_with_checkpoint(path, label):
    state_path = CHECKPOINT_DIR / f"{label.lower()}_state.pkl"

    if state_path.exists():
        with open(state_path, "rb") as f:
            state = pickle.load(f)

        start_row = state["processed_rows"]
        heaps = state["heaps"]

        print(f"{label}: RESUMING at row {start_row:,}")

    else:
        start_row = 0
        heaps = {}
        print(f"{label}: starting from row 0")

    started = time.time()
    processed = 0
    last_checkpoint = start_row

    # heap entry = (score, source_entity_id)
    # We keep the best TOP_K_PER_S1 source rows for each S1.
    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")

        for row_number, row in enumerate(reader):
            if row_number < start_row:
                continue

            processed += 1

            sid2 = str(row.get("entity_id", "")).strip()
            if not sid2:
                continue

            country2 = str(row.get("country", "")).strip().upper()
            name2 = normalize_business_name(row.get("business_name", ""))
            addr2 = normalize_address(row.get("business_address", ""))

            # Candidate positions + number of independent blocking hits.
            candidates = source_candidates(row)

            for pos, key_count in candidates:

                s1 = s1_rows[pos]

                # Country is already embedded in blocking keys, but keep
                # this guard for exact fallback safety.
                if s1["country"] != country2:
                    continue

                s1_name = normalize_business_name(s1["business_name"])
                s1_addr = normalize_address(s1["business_address"])

                cheap = cheap_score_from_normalized(
                    s1_name,
                    s1_addr,
                    name2,
                    addr2,
                    key_count
                )

                if cheap < CHEAP_THRESHOLD:
                    continue

                # Detailed scorer only for strong cheap candidates.
                s2_row = {
                    "entity_id": sid2,
                    "business_name": row.get("business_name", ""),
                    "business_address": row.get("business_address", ""),
                    "country": country2,
                }

                score = detailed_score(s1, s2_row)

                if score < FINAL_THRESHOLD:
                    continue

                sid1 = s1["entity_id"]

                heap = heaps.setdefault(sid1, [])

                item = (float(score), sid2)

                if len(heap) < TOP_K_PER_S1:
                    heapq.heappush(heap, item)
                elif item > heap[0]:
                    heapq.heapreplace(heap, item)

            # ----------------------------------------------------
            # Progress + checkpoint
            # ----------------------------------------------------
            absolute_row = row_number + 1

            if absolute_row % 100_000 == 0:
                elapsed = time.time() - started
                print(
                    f"{label}: {absolute_row:,} rows | "
                    f"S1 with accepted candidates={len(heaps):,} | "
                    f"time={elapsed/60:.1f} min"
                )

            if absolute_row % 250_000 == 0:
                checkpoint = {
                    "processed_rows": absolute_row,
                    "heaps": heaps,
                }

                tmp = state_path.with_suffix(".tmp")

                with open(tmp, "wb") as f:
                    pickle.dump(
                        checkpoint,
                        f,
                        protocol=pickle.HIGHEST_PROTOCOL
                    )

                os.replace(tmp, state_path)
                last_checkpoint = absolute_row

                print(
                    f"  checkpoint saved at {absolute_row:,}"
                )

            if (time.time() - started) >= CELL_TIME_LIMIT_MIN * 60:
                checkpoint = {
                    "processed_rows": absolute_row,
                    "heaps": heaps,
                }

                tmp = state_path.with_suffix(".tmp")

                with open(tmp, "wb") as f:
                    pickle.dump(
                        checkpoint,
                        f,
                        protocol=pickle.HIGHEST_PROTOCOL
                    )

                os.replace(tmp, state_path)

                print()
                print("=" * 72)
                print(f"⏸️ {label} TIME GUARD REACHED")
                print(f"Checkpoint row: {absolute_row:,}")
                print("RERUN THIS SAME CELL TO CONTINUE.")
                print("=" * 72)
                return False

    # Completed.
    checkpoint = {
        "processed_rows": absolute_row,
        "heaps": heaps,
        "complete": True,
    }

    with open(state_path, "wb") as f:
        pickle.dump(
            checkpoint,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    print()
    print("=" * 72)
    print(f"✅ {label} COMPLETE")
    print(f"Rows processed: {absolute_row:,}")
    print(f"S1 with accepted matches: {len(heaps):,}")
    print("=" * 72)

    return True


s2_complete = run_source_with_checkpoint(TEST_S2, "S2")

gc.collect()

## Cell 8 — Full TEST S3 inference with checkpoint/resume

Same controlled procedure as S2.

If the 45-minute guard is reached, rerun this same cell.

In [ ]:
# ============================================================
# CELL 8 — STREAM TEST S3
# ============================================================

s3_complete = run_source_with_checkpoint(TEST_S3, "S3")

gc.collect()

## Cell 9 — Combine S2 + S3 and write the required TSV files

This cell runs only after both source passes are complete.

`candidate_pairs.tsv` contains the final candidate set actually retained
by the final matching stage.

`matching_results.tsv` contains the retained candidates that passed the
final score threshold.

In [ ]:
# ============================================================
# CELL 9 — BUILD FINAL OUTPUT FILES
# ============================================================

S2_STATE = CHECKPOINT_DIR / "s2_state.pkl"
S3_STATE = CHECKPOINT_DIR / "s3_state.pkl"

assert S2_STATE.exists(), "S2 checkpoint missing."
assert S3_STATE.exists(), "S3 checkpoint missing."

with open(S2_STATE, "rb") as f:
    s2_state = pickle.load(f)

with open(S3_STATE, "rb") as f:
    s3_state = pickle.load(f)

assert s2_state.get("complete", False), "S2 is not complete. Finish Cell 7 first."
assert s3_state.get("complete", False), "S3 is not complete. Finish Cell 8 first."

s2_heaps = s2_state["heaps"]
s3_heaps = s3_state["heaps"]

# Merge per-S1 source candidates.
combined = defaultdict(list)

for source_heaps in (s2_heaps, s3_heaps):
    for sid1, heap in source_heaps.items():
        combined[sid1].extend(heap)

# Keep the best TOP_K_PER_S1 across S2+S3.
final_by_s1 = {}

for sid1, entries in combined.items():
    entries = sorted(
        entries,
        key=lambda x: (-x[0], x[1])
    )

    seen = set()
    clean = []

    for score, sid in entries:
        if sid in seen:
            continue
        seen.add(sid)
        clean.append((score, sid))

        if len(clean) >= TOP_K_PER_S1:
            break

    final_by_s1[sid1] = clean

MATCHING_PATH = OUTPUT_DIR / "matching_results.tsv"
CANDIDATE_PATH = OUTPUT_DIR / "candidate_pairs.tsv"

with open(MATCHING_PATH, "w", encoding="utf-8", newline="") as fm,      open(CANDIDATE_PATH, "w", encoding="utf-8", newline="") as fc:

    fm.write("source1_entity_id\tmatched_entity_ids\n")
    fc.write("source1_entity_id\tcandidate_entity_ids\n")

    for sid1 in s1_ids:
        entries = final_by_s1.get(sid1, [])

        # All retained candidates already passed FINAL_THRESHOLD.
        ids = [sid for score, sid in entries]

        fm.write(
            sid1 + "\t" + ",".join(ids) + "\n"
        )

        fc.write(
            sid1 + "\t" + ",".join(ids) + "\n"
        )

print("=" * 72)
print("CELL 9 — OUTPUT COMPLETE")
print("=" * 72)
print("TEST S1 rows:", f"{len(s1_ids):,}")
print("S1 with retained candidates:", f"{len(final_by_s1):,}")
print("Matching:", MATCHING_PATH)
print("Candidate:", CANDIDATE_PATH)
print("Matching size:", f"{MATCHING_PATH.stat().st_size / 1024**2:,.1f} MB")
print("Candidate size:", f"{CANDIDATE_PATH.stat().st_size / 1024**2:,.1f} MB")
print()
print("✅ CELL 9 COMPLETE")

## Cell 10 — Run the official competition validator

This cell does not modify the predictions.

In [ ]:
# ============================================================
# CELL 10 — OFFICIAL VALIDATOR
# ============================================================

import subprocess

VALIDATOR = UTILS_DIR / "validate_submission.py"

result = subprocess.run(
    [
        sys.executable,
        str(VALIDATOR),
        "--matching", str(MATCHING_PATH),
        "--candidate", str(CANDIDATE_PATH),
        "--test-dir", str(TEST_DIR),
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Official validator failed with return code {result.returncode}"
    )

print("✅ OFFICIAL VALIDATION PASSED")

## Cell 11 — Package the final submission

Creates the required ZIP structure.

In [ ]:
# ============================================================
# CELL 11 — PACKAGE FINAL SUBMISSION
# ============================================================

import shutil
import zipfile

PACKAGE_DIR = WORK_DIR / "submission_package"
ZIP_PATH = WORK_DIR / "amazon_ml_final_submission.zip"

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

(PACKAGE_DIR / "output").mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / "code" / "business_entity_resolution" / "src").mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    MATCHING_PATH,
    PACKAGE_DIR / "output" / "matching_results.tsv"
)

shutil.copy2(
    CANDIDATE_PATH,
    PACKAGE_DIR / "output" / "candidate_pairs.tsv"
)

# Copy supplied source implementation.
for p in SRC_DIR.glob("*.py"):
    shutil.copy2(
        p,
        PACKAGE_DIR / "code" / "business_entity_resolution" / "src" / p.name
    )

with open(
    PACKAGE_DIR / "code" / "business_entity_resolution" / "README.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        "# Amazon ML Challenge 2026 — Business Entity Resolution\n\n"
        "Final submission using normalized business-name/address blocking "
        "and two-stage similarity scoring.\n"
    )

with open(
    PACKAGE_DIR / "code" / "business_entity_resolution" / "requirements.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("pandas\n")

with open(
    PACKAGE_DIR / "Documentation_template.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        "# Amazon ML Challenge 2026\n\n"
        "## Candidate generation\n"
        "Multi-key normalized business-name and address blocking.\n\n"
        "## Matching\n"
        "Two-stage structural filtering followed by supplied similarity scoring.\n"
    )

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in PACKAGE_DIR.rglob("*"):
        if p.is_file():
            z.write(
                p,
                p.relative_to(PACKAGE_DIR)
            )

print("=" * 72)
print("PACKAGE CREATED")
print("=" * 72)
print("ZIP :", ZIP_PATH)
print("Size:", f"{ZIP_PATH.stat().st_size / 1024**2:,.1f} MB")
print()
print("✅ CELL 11 COMPLETE")

## Cell 12 — Final sanity check

No inference is performed here. This only checks that the ZIP and required
files exist.

In [ ]:
# ============================================================
# CELL 12 — FINAL SANITY CHECK
# ============================================================

import zipfile

assert MATCHING_PATH.exists()
assert CANDIDATE_PATH.exists()
assert ZIP_PATH.exists()

with zipfile.ZipFile(ZIP_PATH) as z:
    names = set(z.namelist())

required = {
    "output/matching_results.tsv",
    "output/candidate_pairs.tsv",
    "Documentation_template.md",
}

missing = required - names

assert not missing, f"Missing ZIP members: {missing}"

print("=" * 72)
print("FINAL SANITY CHECK PASSED")
print("=" * 72)
print("Matching :", MATCHING_PATH)
print("Candidate:", CANDIDATE_PATH)
print("ZIP      :", ZIP_PATH)
print()
print("🎯 READY TO SUBMIT")